In [4]:
import pandas as pd
import numpy as np
import os

from sqlalchemy import create_engine, text
from sqlalchemy.engine import URL

In [5]:
DB_USER = "postgres"
DB_PASSWORD = "Vikrant123"
DB_HOST = "localhost"
DB_PORT = 5432
DB_NAME = "Supply_Chain_Forecasting"

connection_url = URL.create(
    drivername="postgresql+psycopg2",
    username=DB_USER,
    password=DB_PASSWORD,
    host=DB_HOST,
    port=DB_PORT,
    database=DB_NAME
)

engine = create_engine(connection_url)

In [6]:
print("Engine type:", type(engine))

with engine.connect() as connection:
    result = connection.execute(text("SELECT 1"))
    print("Connection test:", result.scalar())

print("PostgreSQL connected successfully!")

Engine type: <class 'sqlalchemy.engine.base.Engine'>
Connection test: 1
PostgreSQL connected successfully!


In [7]:
query = """
SELECT *
FROM supply_chain.sales;
"""

with engine.connect() as connection:
    df = pd.read_sql(text(query), connection)

print("Dataset Shape:", df.shape)
display(df.head())

Dataset Shape: (3000888, 21)


,id,date,store_nbr,family,sales,onpromotion,city,state,type,cluster,...,transactions,has_holiday_event,holiday_event_count,year,month,day,day_of_week,week_of_year,quarter,is_weekend
0,0,2013-01-01,1,AUTOMOTIVE,0.0,0,Quito,Pichincha,D,13,...,NaN,1,1,2013,1,1,1,1,1,0
1,1,2013-01-01,1,BABY CARE,0.0,0,Quito,Pichincha,D,13,...,NaN,1,1,2013,1,1,1,1,1,0
2,2,2013-01-01,1,BEAUTY,0.0,0,Quito,Pichincha,D,13,...,NaN,1,1,2013,1,1,1,1,1,0
3,3,2013-01-01,1,BEVERAGES,0.0,0,Quito,Pichincha,D,13,...,NaN,1,1,2013,1,1,1,1,1,0
4,4,2013-01-01,1,BOOKS,0.0,0,Quito,Pichincha,D,13,...,NaN,1,1,2013,1,1,1,1,1,0


In [8]:
print("Columns:", df.columns.tolist())

print("\nMissing Values:")
print(df.isnull().sum().sort_values(ascending=False).head(10))

print("\nData Types:")
print(df.dtypes)

Columns: ['id', 'date', 'store_nbr', 'family', 'sales', 'onpromotion', 'city', 'state', 'type', 'cluster', 'dcoilwtico', 'transactions', 'has_holiday_event', 'holiday_event_count', 'year', 'month', 'day', 'day_of_week', 'week_of_year', 'quarter', 'is_weekend']

Missing Values:
dcoilwtico      858924
transactions    245784
store_nbr            0
date                 0
id                   0
sales                0
family               0
onpromotion          0
city                 0
type                 0
dtype: int64

Data Types:
id                       int64
date                       str
store_nbr                int64
family                     str
sales                  float64
onpromotion              int64
city                       str
state                      str
type                       str
cluster                  int64
dcoilwtico             float64
transactions           float64
has_holiday_event        int64
holiday_event_count      int64
year                     int64
m

In [9]:
df["date"] = pd.to_datetime(df["date"], errors="coerce")
df["sales"] = pd.to_numeric(df["sales"], errors="coerce")

df = df.dropna(subset=["date", "sales"]).copy()

df = df.sort_values(
    ["store_nbr", "family", "date"]
).reset_index(drop=True)

print("Prepared dataset:", df.shape)

Prepared dataset: (3000888, 21)


In [10]:
# Check duplicate records
duplicate_count = df.duplicated(
    subset=["date", "store_nbr", "family"]
).sum()

# Check missing calendar days within each store-category series
date_gaps = (
    df.groupby(
        ["store_nbr", "family"],
        sort=False
    )["date"]
    .diff()
    .dt.days
)

gap_count = (date_gaps.dropna() != 1).sum()

print("Duplicate records:", duplicate_count)
print("Non-daily date gaps:", gap_count)
print("Unique stores:", df["store_nbr"].nunique())
print("Product categories:", df["family"].nunique())
print("Date range:", df["date"].min(), "to", df["date"].max())

Duplicate records: 0
Non-daily date gaps: 7128
Unique stores: 54
Product categories: 33
Date range: 2013-01-01 00:00:00 to 2017-08-15 00:00:00


In [11]:
df["year"] = df["date"].dt.year
df["month"] = df["date"].dt.month
df["day"] = df["date"].dt.day
df["day_of_week"] = df["date"].dt.dayofweek
df["day_of_year"] = df["date"].dt.dayofyear
df["quarter"] = df["date"].dt.quarter
df["week_of_year"] = df["date"].dt.isocalendar().week.astype("int16")

df["is_weekend"] = (
    df["day_of_week"] >= 5
).astype("int8")

print("Calendar features created!")

Calendar features created!


In [12]:
import numpy as np

df["month_sin"] = np.sin(
    2 * np.pi * df["month"] / 12
)

df["month_cos"] = np.cos(
    2 * np.pi * df["month"] / 12
)

df["day_sin"] = np.sin(
    2 * np.pi * df["day_of_week"] / 7
)

df["day_cos"] = np.cos(
    2 * np.pi * df["day_of_week"] / 7
)

print("Cyclical features created!")

Cyclical features created!


In [13]:
group_columns = ["store_nbr", "family"]

df = df.sort_values(
    ["store_nbr", "family", "date"]
).reset_index(drop=True)

grouped = df.groupby(
    group_columns,
    sort=False
)["sales"]

for lag in [1, 7, 14, 28]:

    df[f"sales_lag_{lag}"] = grouped.shift(lag)

print("Lag features created!")

Lag features created!


In [14]:
for window in [7, 14, 28]:

    df[f"sales_rolling_mean_{window}"] = (
        df.groupby(
            group_columns,
            sort=False
        )["sales"]
        .transform(
            lambda x: x.shift(1)
            .rolling(
                window=window,
                min_periods=window
            )
            .mean()
        )
    )

print("Rolling averages created!")

Rolling averages created!


In [15]:
feature_check = [
    "date",
    "store_nbr",
    "family",
    "sales",
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_28",
    "sales_rolling_mean_7",
    "sales_rolling_mean_14",
    "sales_rolling_mean_28"
]

display(df[feature_check].head(35))

,date,store_nbr,family,sales,sales_lag_1,sales_lag_7,sales_lag_14,sales_lag_28,sales_rolling_mean_7,sales_rolling_mean_14,sales_rolling_mean_28
0,2013-01-01,1,AUTOMOTIVE,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2013-01-02,1,AUTOMOTIVE,2.0,0.0,NaN,NaN,NaN,NaN,NaN,NaN
2,2013-01-03,1,AUTOMOTIVE,3.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
3,2013-01-04,1,AUTOMOTIVE,3.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN
4,2013-01-05,1,AUTOMOTIVE,5.0,3.0,NaN,NaN,NaN,NaN,NaN,NaN
5,2013-01-06,1,AUTOMOTIVE,2.0,5.0,NaN,NaN,NaN,NaN,NaN,NaN
6,2013-01-07,1,AUTOMOTIVE,0.0,2.0,NaN,NaN,NaN,NaN,NaN,NaN
7,2013-01-08,1,AUTOMOTIVE,2.0,0.0,0.0,NaN,NaN,2.142857,NaN,NaN
8,2013-01-09,1,AUTOMOTIVE,2.0,2.0,2.0,NaN,NaN,2.428571,NaN,NaN
9,2013-01-10,1,AUTOMOTIVE,2.0,2.0,3.0,NaN,NaN,2.428571,NaN,NaN


In [16]:
lag_columns = [
    "sales_lag_1",
    "sales_lag_7",
    "sales_lag_14",
    "sales_lag_28",
    "sales_rolling_mean_7",
    "sales_rolling_mean_14",
    "sales_rolling_mean_28"
]

df_model = df.dropna(
    subset=lag_columns
).copy()

print("Original dataset:", df.shape)
print("Final ML dataset:", df_model.shape)
print("Removed records:", len(df) - len(df_model))

Original dataset: (3000888, 33)
Final ML dataset: (2950992, 33)
Removed records: 49896


In [ ]:
import os

BASE_DIR = r"C:\Users\vikra\Desktop\Supply Chain Algo"

OUTPUT_DIR = os.path.join(
    BASE_DIR,
    "Data",
    "processed"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

df_model.to_csv(
    os.path.join(
        OUTPUT_DIR,
        "sales_features.csv.gz"
    ),
    index=False,
    compression="gzip"
)

print("Feature-engineered dataset saved successfully!")